In [1]:
import sys, os
# My dot keeps going missing
os.environ["PATH"] = "/opt/homebrew/Cellar/graphviz/12.2.1/bin/:" + os.environ["PATH"]
sys.path.insert(0, os.path.abspath(".."))

import polars as pl
from decider import initialize_decider
initialize_decider() # Imports all the submodules

In [2]:
from decider.modules.credit.scorecard.module import ScoreCard, ScoredVariable, ProbabilityDefault
from decider.modules.credit.scorecard.impl import BoundBin, ValuesBin, DefaultBin
from ped.modules.credit.decision_table.module import DecisionTableModule
from ped.modules.credit.decision_table.config import ParametersConfig, BetweenExpression, InExpression, AndExpression

/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "DataFrame" shadows an attribute in parent "BaseModel"
  warnings.warn(
/opt/miniconda/envs/dspdev/lib/python3.10/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "WithTreeOutput" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [3]:
scorecard = ScoreCard(
    name="my_scorecard",
    variables=[
        ScoredVariable(
            name="income_var",
            variable_name="income",
            bins=[
                BoundBin(value=50,  lower_bound=None,   upper_bound=30000),
                BoundBin(value=100, lower_bound=30000,  upper_bound=60000),
                BoundBin(value=150, lower_bound=60000,  upper_bound=100000),
                BoundBin(value=200, lower_bound=100000, upper_bound=None),
            ],
            default=DefaultBin(value=75),
            value_output_name="income_score",
        ),
        ScoredVariable(
            name="dti_var",
            variable_name="debt_to_income",
            bins=[
                BoundBin(value=200, lower_bound=None, upper_bound=0.2),
                BoundBin(value=150, lower_bound=0.2,  upper_bound=0.4),
                BoundBin(value=100, lower_bound=0.4,  upper_bound=0.6),
                BoundBin(value=50,  lower_bound=0.6,  upper_bound=None),
            ],
            default=DefaultBin(value=100),
            value_output_name="dti_score",
        ),
        ScoredVariable(
            name="employment_var",
            variable_name="employment_status",
            bins=[
                ValuesBin(value=150, items=["full_time"]),
                ValuesBin(value=100, items=["part_time"]),
                ValuesBin(value=50,  items=["unemployed"]),
            ],
            default=DefaultBin(value=75),
            value_output_name="employment_score",
        ),
    ],
    score_output_name="score",
)

scorecard

ScoreCard(type='scorecard', name='my_scorecard', variables=[ScoredVariable(type='scored', name='income_var', variable_name='income', bins=[BoundBin(value=50, name=None, lower_bound=None, upper_bound=30000), BoundBin(value=100, name=None, lower_bound=30000, upper_bound=60000), BoundBin(value=150, name=None, lower_bound=60000, upper_bound=100000), BoundBin(value=200, name=None, lower_bound=100000, upper_bound=None)], default=DefaultBin(value=75, name=None), strict=True, raw_output_name=None, value_output_name='income_score', variable_struct_function=None, struct_to_score_function=None), ScoredVariable(type='scored', name='dti_var', variable_name='debt_to_income', bins=[BoundBin(value=200, name=None, lower_bound=None, upper_bound=0.2), BoundBin(value=150, name=None, lower_bound=0.2, upper_bound=0.4), BoundBin(value=100, name=None, lower_bound=0.4, upper_bound=0.6), BoundBin(value=50, name=None, lower_bound=0.6, upper_bound=None)], default=DefaultBin(value=100, name=None), strict=True, raw

In [4]:
pd = ProbabilityDefault(name="pd")

In [5]:
decision_table = DecisionTableModule(
    name="risk_band",
    parameters=ParametersConfig(
        columns=["min_score", "max_score", "risk_label"],
        values=[
            [0,   300, "high"],
            [300, 450, "medium"],
            [450, 600, "low"],
            [600, None, "extremely low"],
        ],
        dtypes={"min_score": "int", "max_score": "Optional[float]", "risk_label": "string"},
    ),
    expression=AndExpression(
        expressions=[BetweenExpression(
            variable="score",
            lower_bound_column="min_score",
            upper_bound_column="max_score",
            mode="lower_inclusive",
        ),
        BetweenExpression(
            variable="external_var",
            lower_bound_column="min_score",
            upper_bound_column="max_score",
            mode="lower_inclusive",
        ),]
    )
    outputs=["risk_label"],
    default=["unknown"],
)


In [ ]:

# ── When every module has exactly one input and one output, | is unambiguous ──

simple_pipeline = scorecard | pd

# ── When a module has multiple inputs, | breaks down ─────────────────────────
# decision_table takes both 'score' (from pd) and 'external_var' (from outside).
# We need a way to say: "wire pd's output into 'score'; leave 'external_var' external."

# Option A: .bind() on the receiving module
# — name only the inputs that come from upstream; the rest stay external.
pipeline = scorecard | pd | decision_table.bind(score=pd)

# Option B: same idea, but when you also need to rename an output before routing
# (e.g. scorecard has multiple outputs and you want to pick one explicitly)
pipeline = (
    scorecard
    | pd.bind(score=scorecard["score"])         # pick the named output from scorecard
    | decision_table.bind(score=pd)             # route pd's output to 'score' input
)                                               # 'external_var' is still external

# Option C: completely explicit — useful when the graph isn't linear
# (fan-out, fan-in, or non-contiguous connections)
pipeline = Chain(
    scorecard,
    pd,
    decision_table.bind(
        score=pd,                # wired from pd
        external_var=External(), # explicitly mark as external input — same as omitting it
    ),
    name="credit_pipeline",
)


## Comparing two API designs

Two approaches for when connections are ambiguous.

### Design A — receiver-centric (`.bind()`)
You configure the *receiving* module's inputs at definition time.
```python
decision_table.bind(score=pd)
```
### Design B — wire-centric (`.outputs | .inputs`)
You describe the *wire* between two named ports.
```python
simple_pipeline.outputs.pd | decision_table.inputs.score
```

---

| | Design A `.bind()` | Design B `.outputs\|.inputs` |
|---|---|---|
| Mental model | "configure this module's inputs" | "draw a wire between two ports" |
| Python familiarity | kwargs are idiomatic Python | more like a node-graph editor (Blender, Max/MSP) |
| Simple linear chain | `scorecard \| pd` (no change) | `scorecard \| pd` (no change) |
| Ambiguous input | `decision_table.bind(score=pd)` | `pd.outputs.probability \| decision_table.inputs.score` |
| Multi-output source | `decision_table.bind(score=pd.outputs.probability)` | `pd.outputs.probability \| decision_table.inputs.score` |
| Fan-in (two sources) | `node.bind(a=mod1, b=mod2)` | `mod1.outputs.x \| node.inputs.a` then `mod2.outputs.y \| node.inputs.b` — two separate statements, harder to read as one unit |
| IDE autocomplete | kwargs hint at input names | `.outputs.` and `.inputs.` are discoverable namespaces ✓ |
| Reads as prose | "bind score to pd" — receiver-focused | "pd's output flows into score" — data-flow-focused |

---

**Key tension:** `.outputs.pd` in Design B is ambiguous — is `pd` the *module name* or the *output variable name*?  
They differ as soon as the module's output variable isn't the same string as the module's name.  
`.outputs.pd.value` makes this worse — during construction you're holding a *reference*, not a value.  

**Verdict:** Both are valid, but for Python they pull in different directions.  
A pragmatic hybrid: keep Design A's `.bind()` for configuring connections, but borrow `.outputs.name` from Design B for *selecting a specific output* within `.bind()`:

```python
# unambiguous, discoverability via .outputs, configured at the receiver
decision_table.bind(
    score=pd.outputs.probability,   # pick the named output explicitly
    # external_var is omitted → stays an external input automatically
)
```


In [ ]:
# Imagine pd outputs two things: probability_of_default and credit_score

# ── Case 1: one output is passed on, the other stays in the graph ────────────
# The `|` shorthand is now ambiguous — which output feeds the next module?
# You have to be explicit on the receiver.

pipeline = (
    scorecard
    | pd                                         # pd has two outputs; | still works to append
    | decision_table.bind(
        score=pd.outputs.credit_score,           # pick the specific output
    )
)

# ── Case 2: both outputs feed different downstream modules (fan-out) ──────────
# | can't express this linearly, so you drop into the explicit form entirely.

pipeline = Chain(
    scorecard,
    pd,
    risk_model.bind(pd=pd.outputs.probability_of_default),
    decision_table.bind(score=pd.outputs.credit_score),
    name="credit_pipeline",
)

# ── Why this holds together ───────────────────────────────────────────────────
# - `pd` in `.bind()` refers to the *module object*, not its name string.
#   Python identity — no magic string matching.
# - `.outputs.credit_score` is only needed when there are >1 outputs.
#   When there is exactly one, `pd` alone is unambiguous and `.outputs` is skipped.
# - The rule: escalate specificity only when needed.
#
#   pd (single output)           → just pass `pd`
#   pd (multi-output)            → pass `pd.outputs.credit_score`
#   nested chain (multi-output)  → pass `simple_pipeline.outputs.credit_score`


In [ ]:
# ── Two more operator proposals ──────────────────────────────────────────────

# Proposal 1: << as a "configure inputs" operator
p = scorecard | pd | (decision_table << {"score": pd.outputs.credit_score})

# Proposal 2: dict on the left of |
p = scorecard | pd
p = {"score": pd.outputs.credit_score, "external_var": pd} | decision_table

# ── Assessment ───────────────────────────────────────────────────────────────

# Proposal 1 — `<<`
# + Reads reasonably: "push this wiring into decision_table"
# + Can be parenthesised inline with | chains, producing one expression
# + No Python conflict — << (__lshift__) is free to override on BaseModule
# - << has no strong semantic anchor in Python; unfamiliar compared to kwargs
# - dict syntax loses IDE autocomplete on key names (strings vs kwargs)

# Proposal 2 — dict | module
# ✗ Blocked by Python itself: in Python 3.9+ dict.__or__ is defined for dict merging.
#   {"score": ...} | decision_table will call dict.__or__ first, return NotImplemented,
#   then fall back to decision_table.__ror__ — BUT only if the right side is not a dict.
#   This actually works via __ror__, but the behaviour is surprising to readers
#   who know that dict | dict is a merge. A linter/reader will expect a dict back.
# - Also: the wiring and the module are on opposite sides of |, so you read
#   right-to-left for the destination and left-to-right for the source — mixed flow.

# ── Verdict ───────────────────────────────────────────────────────────────────
# `<<` is the cleaner operator pick if you want to avoid .bind(),
# but it trades kwargs discoverability for operator novelty.
# The hybrid that holds up best across all cases:

p = scorecard | pd | (decision_table << {"score": pd.outputs.credit_score})
#   ↑ linear flow preserved        ↑ wiring is a self-contained pre-config step

# vs .bind() — identical semantics, more Pythonic
p = scorecard | pd | decision_table.bind(score=pd.outputs.credit_score)


# Implementation Spec: MapperModule + Selector API (decider)

## Context
All work is in **`decider/`**. PED is out of scope.

Decider's `BaseModule` currently has only `name: str` and abstract `expand_nodes() -> List[Node]`.  
Decider's `Node.input_map` is `Dict[str, Union[Node, StaticValueNode, ExternalInputNode]]` — not strings.  
There is no `ChainModule` or `primitives/` directory in decider — nothing to migrate or delete.

---

## New file: `decider/modules/primitives/mapper.py`

### `ModuleOutputSelector`
```python
@dataclass
class ModuleOutputSelector:
    module: BaseModule
    output_node_name: str   # the node name (un-namespaced) that this output refers to

    def __or__(self, other: "ModuleInputSelector") -> "MapperModule":
        """Wire this output directly into a named input of another module."""
        if not isinstance(other, ModuleInputSelector):
            raise TypeError(...)
        return MapperModule(
            name=self.module.name,
            modules=[self.module, other.module],
            mappings={other.module.name: {other.input_name: (self.module.name, self.output_node_name)}},
        )
```

### `ModuleInputSelector`
```python
@dataclass
class ModuleInputSelector:
    module: BaseModule
    input_name: str   # the ExternalInputNode.input_name this selector refers to
```

### `ModuleOutputAccessor` / `ModuleInputAccessor`
Proxy objects returned by `module.outputs` / `module.inputs` properties.
Both support attribute access and item access returning the corresponding selector.
```python
class ModuleOutputAccessor:
    def __init__(self, module: BaseModule): ...
    def __getattr__(self, name: str) -> ModuleOutputSelector: ...
    def __getitem__(self, name: str) -> ModuleOutputSelector: ...

class ModuleInputAccessor:
    def __init__(self, module: BaseModule): ...
    def __getattr__(self, name: str) -> ModuleInputSelector: ...
    def __getitem__(self, name: str) -> ModuleInputSelector: ...
```

### `MapperModule`
```python
class MapperModule(BaseModule):
    type: Literal["mapper"]
    modules: List[BaseModule]

    # {dest_module_name: {input_variable_name: (src_module_name, src_output_node_name)}}
    # "input_variable_name" is the ExternalInputNode.input_name that will be rewired.
    # "src_output_node_name" is the un-namespaced Node.name in the source module.
    mappings: Dict[str, Dict[str, Tuple[str, str]]] = {}

    def expand_nodes(self) -> List[Node]: ...
    def module_namespaced_nodes(self, module_name=None) -> List[Node]:
        # No extra namespace layer — just delegates to expand_nodes()
        return self.expand_nodes()
```

---

## Changes to `decider/modules/core.py` — `BaseModule`

### Add `outputs` / `inputs` properties
```python
@property
def outputs(self) -> "ModuleOutputAccessor": ...

@property
def inputs(self) -> "ModuleInputAccessor": ...
```
No field conflict — decider's `BaseModule` has no `input_name`/`output_name` fields.

### Add `output_names` / `input_names` computed properties
```python
@property
def input_names(self) -> List[str]:
    """All external input variable names this module requires.
    Derived by calling expand_nodes() and collecting all ExternalInputNode.input_name
    values from every node's input_map that are not satisfied by another node in the same module.
    Subclasses may override if expand_nodes() is expensive or unavailable at config time.
    """

@property
def output_names(self) -> List[str]:
    """All output node names this module exposes.
    Default: the un-namespaced names of all nodes that are not referenced
    as a dependency by any other node within the same expanded set.
    Subclasses may override to declare outputs explicitly.
    """
```

### Add `__or__` for implicit wiring
```python
def __or__(self, other: Union["BaseModule", "ModuleInputSelector"]) -> "MapperModule":
    """
    Append `other` to this module's graph.

    If `other` is a bare BaseModule, wiring is IMPLICIT:
      - self must expose exactly one output_name
      - other must expose exactly one input_name
      - An entry is added to mappings automatically
      If either condition fails, raise ValueError with a clear message directing
      the user to use .outputs / .inputs for explicit wiring.

    If `other` is a ModuleInputSelector, wiring is EXPLICIT:
      self must expose exactly one output (same rule as above, but the destination
      input is taken from the selector rather than inferred).

    In both cases: if self is already a MapperModule, extend it in-place
    (add module + mapping entry) rather than nesting.
    """
```

### Add `__lshift__` for batch explicit wiring
```python
def __lshift__(self, mapping: Dict[str, Union["BaseModule", "ModuleOutputSelector"]]) -> "MapperModule":
    """
    Return a MapperModule that wraps self with the given explicit input wiring.

    mapping keys   = input variable names on self (ExternalInputNode.input_name)
    mapping values = either:
        - a bare BaseModule  → resolve to its single output at expand time
        - a ModuleOutputSelector → use the named output directly

    Equivalent to .bind(**mapping). Prefer .bind() in code; << is for operator-chain style.
    """
```

### Add `.bind()` method (preferred alias for `__lshift__`)
```python
def bind(self, **kwargs: Union["BaseModule", "ModuleOutputSelector"]) -> "MapperModule":
    """Same semantics as __lshift__ but with kwargs instead of a dict."""
```

---

## `MapperModule.expand_nodes()` — detailed logic

```
1. Build a registry: {module.name: module} for all modules in self.modules.
   Raise ValueError on duplicate module names.

2. For each module, call module.module_namespaced_nodes() to get its namespaced Nodes.
   Collect all nodes into a flat list, keyed by node_id for O(1) lookup.

3. Apply explicit mappings from self.mappings:
   For (dest_module_name, input_var_name) → (src_module_name, src_output_node_name):
     a. Find the source node: node_id = (src_module_name, src_output_node_name)
     b. In all nodes belonging to dest_module_name, find every node whose input_map
        contains an ExternalInputNode(input_name=input_var_name).
     c. Replace that ExternalInputNode with a direct reference to the source Node object.

4. Validate: every ExternalInputNode still remaining in the flat node list is a genuine
   external input (not a wiring mistake). No further patching is done — these become
   the MapperModule's declared external inputs.

5. Return the flat list.
```

---

## Flattening rule: no nested MapperModules

When `|` or `<<` / `.bind()` is called and the left operand is already a `MapperModule`:
- Add the new module(s) to `self.modules` (after checking for name uniqueness)
- Merge the new `mappings` dict into `self.mappings`
- Return `self.model_copy(update={...})` — **do not wrap in a new MapperModule**

When the right operand is already a `MapperModule` (from a `ModuleOutputSelector | ModuleInputSelector` sub-expression):
- Same: pull its `modules` and `mappings` up into the parent, de-duplicating any modules
  already present (by identity / name).

---

## Resolving bare module refs at expand time (`.bind(score=pd)`)

When a mapping value is a bare `BaseModule` (not a `ModuleOutputSelector`), resolve it during `expand_nodes()`:
```
source_output_names = source_module.output_names
if len(source_output_names) == 1:
    use source_output_names[0]
else:
    raise ValueError(
        f"Module '{source_module.name}' has {len(source_output_names)} outputs "
        f"({source_output_names}). Use module.outputs.<name> to select one explicitly."
    )
```

---

## File layout

```
decider/modules/primitives/
    __init__.py          # registers MapperModule via register_graph_module
    mapper.py            # MapperModule, ModuleOutputSelector, ModuleInputSelector,
                         # ModuleOutputAccessor, ModuleInputAccessor
```

`decider/modules/core.py` — add `outputs`, `inputs`, `output_names`, `input_names`, `__or__`, `__lshift__`, `bind` to `BaseModule`.

---

## Out of scope
- PED / ChainModule — irrelevant
- Serialisation migration — no existing mapper configs exist
